In [1]:
import numpy as np
import torch
from ellipsoid import EllipsoidND
import cvxpy as cp
import numpy as np
from numpy.linalg import eig, inv
from scipy.spatial import ConvexHull
from scipy.optimize import linprog




In [2]:
def compute_rank(matrix):


    """
    Compute the rank of a matrix.

    Parameters:
    matrix (numpy array): Input matrix

    Returns:
    int: Rank of the matrix
    """
    # Replace this with your actual compute_rank function
    return np.linalg.matrix_rank(matrix)


In [3]:
def compute_P_prime(P: torch.Tensor):
    """
    Projects a set of points P onto their affine subspace and returns:
    - P_prime: the coordinates of P in the affine subspace (lower dimension)
    - mapping: a list of tuples (original_index, projected_index)

    Args:
        P (np.ndarray): shape (n_points, d), input points

    Returns:
        P_prime (np.ndarray): shape (n_points, k), projected points in affine subspace
        mapping (list): list of tuples (original_index, projected_index)
    """
    P = np.asarray(P)
    mean = np.mean(P, axis=0)
    P_centered = P - mean

    # Use the provided compute_rank function
    rank = compute_rank(P_centered)
    U, S, Vt = np.linalg.svd(P_centered, full_matrices=False)
    Y = Vt[:rank].T  # d x k

    P_prime = P_centered @ Y
    mapping = [(i, i) for i in range(P.shape[0])]

    return P_prime, mapping



In [4]:

def caratheodory_set(v, P):
    n, d = P.shape
    c = np.zeros(n)
    A_eq = np.vstack([P.T, np.ones(n)])
    b_eq = np.append(v, 1)
    bounds = [(0, 1)] * n
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if res.success:
        lambdas = res.x
        indices = np.where(lambdas > 1e-6)[0]
        if len(indices) > d+1:
            top = np.argsort(lambdas[indices])[-(d+1):]
            indices = indices[top]
        return indices, lambdas[indices]
    else:
        return None, None

In [5]:

def compute_mvee(P, tol=1e-5, max_iter=1000):
    """
    Tính toán Minimum Volume Enclosing Ellipsoid (MVEE) cho tập điểm P (n điểm, d chiều).
    Trả về một đối tượng EllipsoidND.

    Args:
        P (np.ndarray): shape (n_points, d), tập các điểm.
        tol (float): ngưỡng hội tụ.
        max_iter (int): số vòng lặp tối đa.

    Returns:
        EllipsoidND: ellipsoid bao ngoài tối tiểu.
    """
    P = np.asarray(P)
    n_points, d = P.shape

    # Khởi tạo trọng số đều
    u = np.ones(n_points) / n_points

    for _ in range(max_iter):
        # Tính ma trận X(u)
        X = (P.T * u) @ P
        X_inv = np.linalg.inv(X)

        # Tính giá trị M_i cho từng điểm
        M = np.einsum('ij,jk,ik->i', P, X_inv, P)

        # Tìm chỉ số có M lớn nhất
        j = np.argmax(M)
        max_M = M[j]

        # Kiểm tra hội tụ
        if max_M - d <= tol:
            break

        # Cập nhật trọng số
        step_size = (max_M - d - 1) / ((d + 1) * (max_M - 1))
        new_u = (1 - step_size) * u
        new_u[j] += step_size
        u = new_u

    # Tính toán ma trận G và tâm c
    X = (P.T * u) @ P
    c = P.T @ u
    G = np.linalg.inv(X)

    return EllipsoidND(G, c)

In [6]:
def l_infinity_coreset(P):
    """
    Tính toán coreset cho bài toán MVEE trong không gian l-infinity.
    
    Args:
        P (np.ndarray): Mảng shape (n_points, d), các điểm đầu vào.
    
    Returns:
        np.ndarray: Mảng shape (m, d), coreset cho bài toán 
    """

    P_prime, mapping = compute_P_prime(P)
    S = []

    big_ellipsoid = compute_mvee(P_prime)
    small_ellipsoid = big_ellipsoid.shrink_ellipsoid(compute_rank(P_prime))
    # Tính toán các đỉnh của ellipsoid nhỏ
    vertices = small_ellipsoid.vertices
    
    
    # Lấy các đỉnh của conv(P_prime)
    hull = ConvexHull(P_prime)
    P_hull = P_prime[hull.vertices]
    

    # Vẽ Carathéodory set cho từng đỉnh của ellipsoid nhỏ chỉ với conv(P_prime)
    caratheodory_results = []
    
   
    colors = ['red', 'green', 'orange', 'purple']
    for i, v in enumerate(vertices):
        idxs, lambdas = caratheodory_set(v, P_hull)
        caratheodory_results.append((v, idxs, lambdas))
        # Vẽ điểm v

        if idxs is not None:
            pts =P_hull[idxs]
 
            # Nối các điểm Carathéodory với v
            for id in idxs:
                S.append(id)
            S = list(set(S))    

        
    return np.array(S)

In [30]:
import torch

def Coreset(P: torch.Tensor, m: int) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Tính toán coreset cho bài toán MVEE trong không gian l-infinity.
    
    Args:
        P (torch.Tensor): Mảng shape (n_points, d), các điểm đầu vào.
        m (int): Kích thước của coreset.
    
    Returns:
        torch.Tensor: Mảng shape (m, d), coreset cho bài toán.
        torch.Tensor: Điểm sensitive của các điểm.
    """
    # Kiểm tra đầu vào


    Q = P.clone()
    s = torch.zeros(P.shape[0], dtype=torch.float32)
    indices = torch.arange(P.shape[0])  # Theo dõi index gốc trong P
    
    i = 1
    l = Q.shape[0]
    r = compute_rank(Q)
    condition = 2 * (r ** 2)
    
    while l >= condition:
        print(f"loop {i}:")
        # Tìm coreset (S là tập index trong Q)
        S = l_infinity_coreset(Q)

        
        # Tính điểm sensitive cho các điểm trong S
        current_rank = compute_rank(Q)
        sensitive_score = current_rank / i
        s[indices[S]] = sensitive_score  # Gán dựa trên index gốc trong P
        
        # Tạo mask để loại bỏ các điểm trong S
        mask = torch.ones(Q.size(0), dtype=torch.bool)
        mask[S] = False
        
        # Cập nhật Q và indices
        Q = Q[mask]
        indices = indices[mask]
        
        l = Q.shape[0]
        r = compute_rank(Q)
        condition = 2 * (r ** 2)
        i += 1
    
    # Tính điểm sensitive cho các điểm còn lại trong Q
    if Q.shape[0] > 0:
        s[indices] = current_rank / i  # Gán dựa trên index gốc trong P
    
    # Chuẩn hóa điểm sensitive
    if s.sum() != 0:
        s = s / s.sum()
    else:
        s = torch.ones_like(s) / s.shape[0]  # Phân phối đều nếu tổng bằng 0
    
    # Chọn m điểm có điểm sensitive cao nhất
    _, top_indices = torch.topk(s, min(m, P.shape[0]))
    
    return P[top_indices], s

In [10]:


# Tạo dữ liệu test với 100 điểm trong không gian 3 chiều
np.random.seed(42)
n_points = 100
n_dim = 3

# Tạo ma trận điểm ngẫu nhiên
points = np.random.randn(n_points, n_dim)
points = torch.tensor(points, dtype=torch.float32)

# Kích thước coreset mong muốn
m = 10

print("=== Test Coreset Generation ===")
print(f"Input shape: {points.shape}")


=== Test Coreset Generation ===
Input shape: torch.Size([100, 3])


In [11]:
rank=compute_rank(points)
print(f"Rank of input points: {rank}")

Rank of input points: 3


In [12]:
# Test các hàm riêng lẻ
print("\nTesting compute_P_prime:")
P_prime, mapping = compute_P_prime(points)
print(f"P_prime shape: {P_prime.shape}")
print(f"First few mappings: {mapping[:5]}")


Testing compute_P_prime:
P_prime shape: (100, 3)
First few mappings: [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4)]


In [13]:
print("\nTesting compute_mvee:")
mvee = compute_mvee(P_prime)
print(f"MVEE center shape: {mvee.c.shape}")
print(f"MVEE matrix G shape: {mvee.G.shape}")


Testing compute_mvee:
MVEE center shape: (3,)
MVEE matrix G shape: (3, 3)


In [14]:
print("\nTesting l_infinity_coreset:")
S = l_infinity_coreset(points)
print(f"l_infinity coreset indices: {S}")
print(f"Number of points in l_infinity coreset: {len(S)}")


Testing l_infinity_coreset:
l_infinity coreset indices: [ 9  2 13 14]
Number of points in l_infinity coreset: 4


In [32]:
# print("\nTesting full Coreset:")
a, b = Coreset(points, m)
print(f"Coreset shape: {a.shape}")
print(f"Number of non-zero sensitive scores: {(b > 0).sum()}")
print(f"Sum of sensitive scores: {b.sum():.4f}")  # Should be close to 1.0

loop 1:
loop 2:
loop 3:
loop 4:
loop 5:
loop 6:
loop 7:
loop 8:
loop 9:
loop 10:
loop 11:
loop 12:
loop 13:
loop 14:
Coreset shape: torch.Size([10, 3])
Number of non-zero sensitive scores: 100
Sum of sensitive scores: 1.0000


In [33]:
a

tensor([[-0.1156, -0.3011, -1.4785],
        [ 0.1969,  0.7385,  0.1714],
        [ 1.5792,  0.7674, -0.4695],
        [ 0.3757, -0.6006, -0.2917],
        [ 0.2089, -1.9597, -1.3282],
        [ 1.5230, -0.2342, -0.2341],
        [-0.3851, -0.6769,  0.6117],
        [ 0.2420, -1.9133, -1.7249],
        [-1.0577,  0.8225, -1.2208],
        [ 1.0310,  0.9313, -0.8392]])

In [34]:
b

tensor([0.0171, 0.0257, 0.0514, 0.0064, 0.0257, 0.0103, 0.0086, 0.0171, 0.0073,
        0.0514, 0.0047, 0.0257, 0.0257, 0.0514, 0.0514, 0.0129, 0.0171, 0.0257,
        0.0257, 0.0073, 0.0257, 0.0257, 0.0086, 0.0171, 0.0171, 0.0171, 0.0129,
        0.0129, 0.0051, 0.0129, 0.0129, 0.0057, 0.0129, 0.0103, 0.0073, 0.0086,
        0.0064, 0.0103, 0.0103, 0.0037, 0.0064, 0.0086, 0.0086, 0.0047, 0.0073,
        0.0073, 0.0073, 0.0073, 0.0057, 0.0057, 0.0064, 0.0064, 0.0057, 0.0051,
        0.0064, 0.0057, 0.0057, 0.0057, 0.0051, 0.0057, 0.0051, 0.0040, 0.0051,
        0.0051, 0.0047, 0.0051, 0.0047, 0.0047, 0.0047, 0.0047, 0.0043, 0.0047,
        0.0043, 0.0040, 0.0043, 0.0043, 0.0043, 0.0043, 0.0037, 0.0037, 0.0040,
        0.0040, 0.0040, 0.0037, 0.0037, 0.0037, 0.0034, 0.0037, 0.0034, 0.0034,
        0.0034, 0.0034, 0.0034, 0.0034, 0.0034, 0.0034, 0.0034, 0.0034, 0.0034,
        0.0034])